# MCDM Sensitivity Analysis — CO₂ Storage Sites

**Companion notebook to the CO₂ storage-capacity studies.** This interactive teaching example ranks three *fictional* storage sites using a weighted sum and tests the sensitivity to criterion weights.

**Reference:** Marco Dean (2022), *A Practical Guide to Multi-Criteria Analysis*, §5.8 (Sensitivity Analysis), including Figure 31. The 1–5 scores, site names, and proportional weight-rescaling rule below are **illustrative choices**, not measurements from Dean or EGDI.

**Workflow:** Runtime → Run all. Change site scores or baseline weights in **Editable inputs**, then rerun downstream cells. All notebook cells are self-contained; no GitHub package installation is needed.

In [ ]:
#@title Import libraries { display-mode: "form" }
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


## 1. Editable inputs

All scores are on a **1–5 desirability scale (5 = best)**. Thus, **Storage cost** is reverse-scored: a *lower actual cost* receives a *higher* desirability score. Site data below are fictional.

Set baseline weights in percentage points; they must be nonnegative and sum to **100%**. The sensitivity grid changes one criterion at a time and rescales the other two weights proportionally.

In [ ]:
#@title Editable inputs { display-mode: "both" }
capacity_weight_pct = 40 #@param {type:"number"}
injectivity_weight_pct = 35 #@param {type:"number"}
cost_weight_pct = 25 #@param {type:"number"}
sensitivity_step_pct = 5 #@param {type:"number"}

# Fictional scores: each row is a site; each column is a criterion.
# You can replace these scores with validated 1–5 desirability scores.
site_scores = pd.DataFrame(
    {
        "Storage capacity": [5, 3, 4],
        "Injectivity": [3, 5, 2],
        "Storage cost": [2, 4, 5],
    },
    index=["Site A", "Site B", "Site C"],
)
criteria = list(site_scores.columns)
weights_pct = np.array([capacity_weight_pct, injectivity_weight_pct, cost_weight_pct], dtype=float)
if np.any(~np.isfinite(weights_pct)) or np.any(weights_pct < 0) or not np.isclose(weights_pct.sum(), 100):
    raise ValueError("Baseline weights must be finite, nonnegative, and sum to 100%.")
numeric_scores = site_scores.to_numpy(dtype=float)
if not np.all(np.isfinite(numeric_scores)) or not np.all((numeric_scores >= 1) & (numeric_scores <= 5)):
    raise ValueError("All scores must be numeric and between 1 and 5.")
if sensitivity_step_pct <= 0 or sensitivity_step_pct > 100 or not np.isclose(100 / sensitivity_step_pct, round(100 / sensitivity_step_pct)):
    raise ValueError("Sensitivity step must divide 100 exactly (e.g. 1, 2, 5, 10).")
weights = weights_pct / 100
print("Fictional desirability scores (5 = best):")
display(site_scores)
display(pd.DataFrame({"Criterion": criteria, "Weight (%)": weights_pct}))
print("Editable inputs validated successfully.")


## 2. Baseline weighted ranking

Weighted sum: $S_i = \\sum_j w_j s_{ij}$.

**Ties are retained** when scores match; a displayed sort order is not an assertion that one tied site is better.

In [ ]:
def weighted_results(score_table, weight_vector):
    if not np.isclose(np.sum(weight_vector), 1) or np.any(weight_vector < -1e-12):
        raise ValueError("Weights must sum to one and be nonnegative.")
    values = score_table.to_numpy(dtype=float) @ np.asarray(weight_vector, dtype=float)
    result = pd.DataFrame({"Site": score_table.index, "Total score": values})
    result["Rank"] = result["Total score"].rank(method="min", ascending=False).astype(int)
    return result.sort_values(["Total score", "Site"], ascending=[False, True]).reset_index(drop=True)

baseline = weighted_results(site_scores, weights)
display(baseline.round(3))
ax = baseline.plot.bar(x="Site", y="Total score", legend=False, figsize=(7, 4))
ax.set_ylabel("Weighted score (1–5)")
ax.set_ylim(0, 5)
ax.set_title("Baseline ranking — fictional scores")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 3. One-at-a-time sensitivity analysis (OAT)

Vary each criterion weight from 0% to 100%, keeping the remaining criterion weights **in their original proportion**. This guarantees that the weights continue to sum to 100%.

The aim is to detect changes in the weighted scores, first-place site(s), and ordering. Grid resolution depends on `sensitivity_step_pct`.

In [ ]:
def rescale_weights(base_weights, index, tested_weight):
    if not 0 <= tested_weight <= 1:
        raise ValueError("Tested weight must be between 0 and 1.")
    original_rest = 1 - base_weights[index]
    if original_rest <= 1e-12:
        if np.isclose(tested_weight, 1):
            result = np.zeros_like(base_weights, dtype=float)
            result[index] = 1
            return result
        raise ValueError("Cannot proportionally rescale other zero baseline weights.")
    result = np.asarray(base_weights, dtype=float) * ((1 - tested_weight) / original_rest)
    result[index] = tested_weight
    return result

rows = []
steps = int(round(100 / sensitivity_step_pct))
baseline_order = baseline["Site"].tolist()
for j, criterion in enumerate(criteria):
    for k in range(steps + 1):
        tested_weight = k / steps
        trial_weights = rescale_weights(weights, j, tested_weight)
        trial = weighted_results(site_scores, trial_weights)
        best = trial["Total score"].max()
        winners = sorted(trial.loc[np.isclose(trial["Total score"], best), "Site"].tolist())
        rows.append({
            "Criterion varied": criterion,
            "Tested weight (%)": 100 * tested_weight,
            **{f"{name} weight (%)": 100 * trial_weights[i] for i, name in enumerate(criteria)},
            **{f"{site} score": float(trial.loc[trial["Site"].eq(site), "Total score"].iloc[0]) for site in site_scores.index},
            "Winner(s)": " / ".join(winners),
            "Display order": " > ".join(trial["Site"]),
            "Order changed vs baseline": trial["Site"].tolist() != baseline_order,
        })
sensitivity = pd.DataFrame(rows)
display(sensitivity.head(10).round(2))
print(f"Evaluated {len(sensitivity)} scenarios ({len(criteria)} criteria × {steps+1} tested weights).")


## 4. Sensitivity curves — Dean-style weight variation

Each chart varies one weight and plots the score of each site. Where curves cross, the preferred ranking may change. The vertical dashed line marks the baseline weight.

In [ ]:
fig, axes = plt.subplots(1, len(criteria), figsize=(17, 4.5), sharey=True)
for j, (criterion, ax) in enumerate(zip(criteria, axes)):
    subset = sensitivity.loc[sensitivity["Criterion varied"] == criterion]
    for site in site_scores.index:
        ax.plot(subset["Tested weight (%)"], subset[f"{site} score"], label=site, linewidth=2)
    ax.axvline(weights_pct[j], linestyle="--", color="gray", linewidth=1, label="Baseline")
    ax.set_title(criterion)
    ax.set_xlabel("Weight of selected criterion (%)")
    ax.set_xlim(0, 100)
    ax.grid(alpha=.25)
axes[0].set_ylabel("Weighted overall score (1–5)")
axes[-1].legend(loc="best", fontsize=9)
plt.suptitle("One-at-a-time weight sensitivity — fictional sites")
plt.tight_layout()
plt.show()


## 5. Robustness summary and output

An ordering change can be caused by a real crossing or by near-ties; interpret the charts and **Winner(s)** column together. Results here are sensitivity to **weights**, not probabilistic uncertainty in reservoir parameters.

The CSV is saved to the Colab runtime's file system. You can download it with the optional final cell.

In [ ]:
summary = (
    sensitivity.groupby("Criterion varied", sort=False)
    .agg(
        Scenarios=("Order changed vs baseline", "size"),
        Changed_order=("Order changed vs baseline", "sum"),
        Different_first_place_sets=("Winner(s)", lambda s: int(sum(x != " / ".join(sorted(baseline.loc[np.isclose(baseline["Total score"], baseline["Total score"].max()), "Site"].tolist())) for x in s))),
    )
    .reset_index()
)
display(summary)
output_file = "mcdm_sensitivity_results.csv"
sensitivity.to_csv(output_file, index=False)
print(f"Saved: {output_file}")


In [ ]:
#@title Optional: download CSV in Colab { display-mode: "form" }
try:
    from google.colab import files
    files.download(output_file)
except ImportError:
    print(f"Not running in Colab. The CSV is saved as {output_file}.")


## Interpretation and limitations

- If the top-ranked site changes at plausible weights, the choice is **weight-sensitive**. If it remains stable, that is evidence of robustness *within the tested weight scenarios only*.
- Weighted sums permit compensation: a high capacity score could mask unacceptable containment or well-integrity risks. Apply **non-compensatory safety / regulatory screening before ranking**.
- Criterion independence, comparable 1–5 scoring rules, and double counting should be checked before using actual site data.
- This example does **not** propagate uncertainty in measured inputs, adjust scoring thresholds, or reproduce a Monte Carlo capacity analysis. These are possible separate extensions.

**Method reference:** Dean, M. (2022), *A Practical Guide to Multi-Criteria Analysis*, §5.8, pp. 69–71.